# Casys ORM - Entity Framework Style

This notebook demonstrates the Entity Framework-style ORM with:
- Entity definitions with relationships
- CRUD operations
- Lazy loading
- Navigation properties
- Query building with lambdas

In [1]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'python'))

import casys_db
from casys_db import NodeEntity, HasMany, HasOne, Session
import tempfile

## 1. Define Entity Models

In [2]:
class Person(NodeEntity):
    """Person entity with relationships."""
    labels = ["Person"]
    name: str
    age: int
    friends = HasMany("Person", via="KNOWS")
    city = HasOne("City", via="LIVES_IN")


class City(NodeEntity):
    """City entity."""
    labels = ["City"]
    name: str
    residents = HasMany("Person", via="LIVES_IN", inverse=True)

print("✓ Entity models defined")

✓ Entity models defined


## 2. Setup Database and Session

In [3]:
# Initialize
tmpdir = tempfile.mkdtemp()
engine = casys_db.CasysEngine(tmpdir)
engine.open_database("social")

try:
    engine.create_branch("social", "main")
except:
    pass

branch = engine.open_branch("social", "main")
session = Session(branch)

print(f"✓ Session created (data dir: {tmpdir})")

✓ Session created (data dir: /tmp/tmp54xmrd02)


## 3. CRUD Operations

In [4]:
# Create entities
alice = Person(name="Alice", age=30)
bob = Person(name="Bob", age=25)
charlie = Person(name="Charlie", age=35)

print("✓ Entities created in memory")
print(f"  Alice: {alice.name}, {alice.age} years old")
print(f"  Bob: {bob.name}, {bob.age} years old")
print(f"  Charlie: {charlie.name}, {charlie.age} years old")

✓ Entities created in memory
  Alice: Alice, 30 years old
  Bob: Bob, 25 years old
  Charlie: Charlie, 35 years old


In [5]:
# Save entities to database
alice_id = session.save(alice)
bob_id = session.save(bob)
charlie_id = session.save(charlie)

print("✓ Entities saved to database")
print(f"  Alice ID: {alice_id}")
print(f"  Bob ID: {bob_id}")
print(f"  Charlie ID: {charlie_id}")

✓ Entities saved to database
  Alice ID: 1
  Bob ID: 2
  Charlie ID: 3


## 4. Query with Fluent API

In [6]:
# Simple query
query = session.query(Person).where(lambda p: p.age > 20)
print("Generated GQL:")
print(f"  {query._build_gql()}")
print(f"\nResults: {query.all()}")

Generated GQL:
  MATCH (p:Person) WHERE p.age > 20 RETURN p

Results: [<__main__.Person object at 0x7f9388e6ee00>, <__main__.Person object at 0x7f9388e6cdf0>, <__main__.Person object at 0x7f9388e6e860>]


In [7]:
# Query with ORDER BY
query = session.query(Person).order_by_desc(lambda p: p.age).take(5)
print("Generated GQL:")
print(f"  {query._build_gql()}")
print(f"\nResults: {query.all()}")

Generated GQL:
  MATCH (p:Person) RETURN p ORDER BY p.age DESC LIMIT 5

Results: [<__main__.Person object at 0x7f9388e6e830>, <__main__.Person object at 0x7f9388e6d5a0>, <__main__.Person object at 0x7f9388e6d0c0>]


In [8]:
# Query with multiple conditions
query = session.query(Person).where(lambda p: p.age > 20 and p.name == 'Alice')
print("Generated GQL:")
print(f"  {query._build_gql()}")
print(f"\nResults: {query.all()}")

Generated GQL:
  MATCH (p:Person) WHERE p.age > 20 AND p.name = 'Alice' RETURN p

Results: [<__main__.Person object at 0x7f9388e6c880>]


In [9]:
# Count query
count = session.query(Person).count()
print(f"Total people: {count}")

Total people: 3


## 5. Traversals with Depth

Demonstrate depth control for relationship traversals.

In [10]:
# Friends up to 3 hops away
query = session.query(Person).where(lambda p: p.name == 'Alice')
query_with_depth = query.depth(3)  # 1 to 3 hops

print("Query with depth(3):")
print(f"  Depth range: {query_with_depth._depth_min} to {query_with_depth._depth_max}")
print(f"  This would generate: MATCH (p:Person)-[:KNOWS*1..3]->(f:Person)")

Query with depth(3):
  Depth range: 3 to 3
  This would generate: MATCH (p:Person)-[:KNOWS*1..3]->(f:Person)


In [11]:
# Friends of friends only (2 to 2 hops)
query_fof = session.query(Person).depth(2, 2)

print("Query with depth(2, 2):")
print(f"  Depth range: {query_fof._depth_min} to {query_fof._depth_max}")
print(f"  This would generate: MATCH (p:Person)-[:KNOWS*2..2]->(f:Person)")

Query with depth(2, 2):
  Depth range: 2 to 2
  This would generate: MATCH (p:Person)-[:KNOWS*2..2]->(f:Person)


In [12]:
# Include starting node (0 to 2 hops)
query_with_self = session.query(Person).depth(0, 2)

print("Query with depth(0, 2):")
print(f"  Depth range: {query_with_self._depth_min} to {query_with_self._depth_max}")
print(f"  This would generate: MATCH (p:Person)-[:KNOWS*0..2]->(f:Person)")
print(f"  (includes the person themselves)")

Query with depth(0, 2):
  Depth range: 0 to 2
  This would generate: MATCH (p:Person)-[:KNOWS*0..2]->(f:Person)
  (includes the person themselves)


## 6. Persistence

In [13]:
# Flush to disk
session.flush()
print("✓ Data flushed to disk")

# Load from disk
session.load()
print("✓ Data loaded from disk")

✓ Data flushed to disk
✓ Data loaded from disk


## 7. Cleanup

In [14]:
import shutil
shutil.rmtree(tmpdir)
print("✓ Cleaned up")

✓ Cleaned up


## Summary

This notebook demonstrated:

1. **Entity Definitions**: `NodeEntity` with `HasMany` and `HasOne` relationships
2. **CRUD**: `session.save()` to create nodes
3. **Fluent Queries**: Lambda-based WHERE, ORDER BY, LIMIT
4. **Depth Control**: `depth(3)`, `depth(2, 5)`, `depth(0, 2)` for traversals
5. **Persistence**: `flush()` and `load()` operations

### Next Steps

- Lazy loading of relationships (when `person.friends` is accessed)
- Creating edges between nodes
- More complex queries with aggregations